In [1]:
using Cosmology
using Jens
using Jens.LensModel.ComLens: CombinedLens
using Jens.LensModel: SIS, Shear
using Jens.LightModel: PointImage
using Jens.LensGenerator: LensedPlane, LightPlane, GenGrid
using Jens.LensSystem: ForwardModel
using Jens.TimeDelay: LensTimeDelay, image_time_delays
using Jens.LensSolver: solve_images
using Jens.LensConstants: DAY_TO_SEC

## cosmology + lens model + grid

In [9]:
cosmo = Cosmology.FlatLCDM(0.7, 0.3, 0.0, 0.0)
z_lens, z_src = 0.3, 1.5

lens = CombinedLens(
    SIS   => (theta_E=1.0, xcentre=0.0, ycentre=0.0),
    Shear => (gamma1=0.05, gamma2=-0.02, xcentre=0.0, ycentre=0.0),
)
lp = LensedPlane(lens; z_lens=z_lens, cosmology=cosmo)

grid = GenGrid(pix_n=128, pix_size=0.09)
xg, yg = grid.xg, grid.yg
beta_x, beta_y = 0.05, -0.03;

## Grid delay from LensedPlane

In [3]:
dt_map = LensTimeDelay(xg, yg, [beta_x, beta_y];
                       LensModel=lp, z_source=z_src);
@show size(dt_map)
@show minimum(dt_map) / DAY_TO_SEC, maximum(dt_map) / DAY_TO_SEC

size(dt_map) = (129, 129)
(minimum(dt_map) / DAY_TO_SEC, maximum(dt_map) / DAY_TO_SEC) = (-31.286453144239694, 2242.122292036941)


(-31.286453144239694, 2242.122292036941)

## Grid delay from LensSystem

In [4]:
agn = PointImage(flux=100.0, beta_x=beta_x, beta_y=beta_y)
sys = ForwardModel(
    lens_plane   = LensedPlane(lens; z_lens=z_lens, cosmology=cosmo),
    source_plane = LightPlane(agn; z=z_src),
    grid         = grid,
)

dt_map2 = LensTimeDelay(sys);
@show all(abs.(dt_map2 .- dt_map) .< 1e-10)

all(abs.(dt_map2 .- dt_map) .< 1.0e-10) = true


true

## Point-wise delay

In [5]:
images = solve_images(sys, beta_x, beta_y; z_source=z_src)

# Scalar
for (tx, ty, mu) in images
    dt = LensTimeDelay(sys, tx, ty)
    println("  ($(round(tx, digits=4)), $(round(ty, digits=4)))  " *
            "Δt = $(round(dt / DAY_TO_SEC, digits=3)) days")
end

# Vector
tx_all = [t[1] for t in images]
ty_all = [t[2] for t in images]
dt_all = LensTimeDelay(sys, tx_all, ty_all);
println("  all: $(round.(dt_all ./ DAY_TO_SEC, digits=3)) days")

  (0.8422, -0.2997)  Δt = -31.316 days
  (-0.3363, 0.6622)  Δt = -22.745 days
  all: [-31.316, -22.745] days


## `image_time_delays` (solve + delay)

In [7]:
images, delays = image_time_delays(sys, beta_x, beta_y)

println("Found $(length(images)) images:")
for i in 1:length(images)
    tx, ty, mu = images[i]
    println("  $i: ($(round(tx, digits=4)), $(round(ty, digits=4)))  " *
            "μ=$(round(mu, digits=1))  Δt=$(round(delays[i] / DAY_TO_SEC, digits=3)) days")
end

println("\nPairwise delays:")
for i in 1:length(delays), j in i+1:length(delays)
    dt = abs(delays[i] - delays[j])
    println("  Δt($i -> $j) = $(round(dt / DAY_TO_SEC, digits=3)) days")
end

Found 2 images:
  1: (0.8422, -0.2997)  μ=7.1  Δt=-31.316 days
  2: (-0.3363, 0.6622)  μ=-11.0  Δt=-22.745 days

Pairwise delays:
  Δt(1 -> 2) = 8.57 days
